# Modelaje financiero y observabilidad

Se entrena un modelo XGBoost para estimar la puntuación crediticia
registrada a partir de las características financieras del cliente.

Se compara su desempeño con una predicción basada en la mediana.

También se implementa una simulación Monte Carlo para explorar escenarios
de ingresos, gastos y saldo durante los siguientes meses.

La estimación del score no representa una probabilidad de incumplimiento.
Los escenarios financieros dependen de supuestos explícitos y no
constituyen pronósticos validados.

Las ejecuciones, métricas, supuestos, errores y alertas se conservan
en una tabla SQLite.

In [9]:
# Importar librerías

from pathlib import Path
from datetime import datetime, timezone
from time import perf_counter
from uuid import uuid4

import hashlib
import json
import sqlite3

import numpy as np
import pandas as pd
import xgboost

from IPython.display import display
from sklearn.dummy import DummyRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor


# Definir rutas

PROYECTO = Path("..").resolve()

RUTA = PROYECTO / "data/processed/gold/customer_financial_summary.parquet"

BD = PROYECTO / "data/observabilidad.sqlite"


# Cargar información

datos = (
    pd.read_parquet(RUTA)
    .sort_values("id")
    .reset_index(drop=True)
)

if datos["id"].isna().any() or datos["id"].duplicated().any():
    raise ValueError(
        "Se requiere una fila por cliente con ID válido."
    )


# Identificar la versión de los datos utilizados

huella = hashlib.sha256(
    pd.util.hash_pandas_object(
        datos,
        index=False
    ).values.tobytes()
).hexdigest()


# Registrar eventos de modelaje

def registrar(evento, estado, detalle, cliente_id=None):
    fila = pd.DataFrame([{
        "evento_id": uuid4().hex,
        "fecha_utc": datetime.now(timezone.utc).isoformat(),
        "evento": evento,
        "estado": estado,
        "cliente_id": cliente_id,
        "datos_sha256": huella,
        "detalle_json": json.dumps(
            detalle,
            ensure_ascii=False,
            allow_nan=False
        )
    }])

    with sqlite3.connect(BD) as conexion:
        fila.to_sql(
            "eventos_modelaje",
            conexion,
            if_exists="append",
            index=False
        )


print("Clientes:", len(datos))
print("XGBoost:", xgboost.__version__)

Clientes: 2037
XGBoost: 3.2.0


## Entrenar y evaluar XGBoost

Se separan los clientes en entrenamiento, validación y prueba.

La validación determina cuándo detener el entrenamiento.
El conjunto de prueba se reserva para calcular las métricas finales.

Se excluyen los identificadores y los límites de crédito, porque la
imputación de esos límites utilizó la puntuación crediticia que ahora
queremos estimar.

Los valores ausentes se conservan como información no disponible.

Se calculan MAE, RMSE y R². Si XGBoost no mejora el MAE de la mediana,
se registra una alerta.

In [10]:
# Definir variables predictoras

variables = [
    "yearly_income",
    "per_capita_income",
    "total_debt",
    "num_credit_cards",
    "total_transactions",
    "total_spent",
    "average_transaction",
    "minimum_transaction",
    "maximum_transaction",
    "total_merchant_cities",
    "has_transaction_records"
]


# Entrenar y registrar resultados

def entrenar():
    inicio = perf_counter()

    try:
        X = datos[variables].astype(float)
        y = datos["credit_score"].astype(float)

        if (
            not np.isfinite(y).all()
            or np.isinf(X.to_numpy()).any()
        ):
            raise ValueError(
                "Hay valores no finitos que requieren revisión."
            )

        # Separar clientes: 60 % entrenamiento,
        # 20 % validación y 20 % prueba

        X_desarrollo, X_test, y_desarrollo, y_test = (
            train_test_split(
                X,
                y,
                test_size=0.20,
                random_state=42
            )
        )

        X_train, X_val, y_train, y_val = train_test_split(
            X_desarrollo,
            y_desarrollo,
            test_size=0.25,
            random_state=42
        )

        # Configurar modelo

        parametros = dict(
            objective="reg:squarederror",
            eval_metric="mae",
            n_estimators=500,
            max_depth=3,
            learning_rate=0.03,
            min_child_weight=10,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=10,
            tree_method="hist",
            early_stopping_rounds=30,
            random_state=42,
            n_jobs=4
        )

        modelo = XGBRegressor(**parametros)

        modelo.fit(
            X_train,
            y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )

        # Crear referencia

        referencia = DummyRegressor(strategy="median")

        referencia.fit(X_train, y_train)

        # Calcular métricas

        filas = []

        for nombre, real, pred in [
            (
                "Mediana prueba",
                y_test,
                referencia.predict(X_test)
            ),
            (
                "XGBoost prueba",
                y_test,
                modelo.predict(X_test)
            ),
            (
                "XGBoost entrenamiento",
                y_train,
                modelo.predict(X_train)
            )
        ]:
            filas.append({
                "modelo": nombre,
                "MAE": float(
                    mean_absolute_error(real, pred)
                ),
                "RMSE": float(
                    np.sqrt(mean_squared_error(real, pred))
                ),
                "R2": float(
                    r2_score(real, pred)
                )
            })

        metricas = pd.DataFrame(filas)

        # Generar alertas

        mejora = filas[1]["MAE"] < filas[0]["MAE"]

        alertas = (
            []
            if mejora
            else ["XGBOOST_NO_SUPERA_MEDIANA"]
        )

        # Guardar información de la ejecución

        detalle = {
            "objetivo": (
                "credit_score registrado; "
                "no probabilidad de impago"
            ),
            "version_xgboost": xgboost.__version__,
            "parametros": parametros,
            "variables": variables,
            "mejor_iteracion": int(modelo.best_iteration),
            "filas_train": len(X_train),
            "filas_validacion": len(X_val),
            "filas_prueba": len(X_test),
            "nulos_por_variable": (
                X.isna().sum().astype(int).to_dict()
            ),
            "metricas": filas,
            "alertas": alertas,
            "duracion_segundos": perf_counter() - inicio
        }

        registrar(
            "entrenamiento",
            "OK" if mejora else "ALERTA",
            detalle
        )

        return modelo, metricas, alertas

    except Exception as error:
        registrar(
            "entrenamiento",
            "ERROR",
            {
                "tipo_error": type(error).__name__,
                "mensaje": str(error),
                "duracion_segundos": perf_counter() - inicio
            }
        )
        raise


modelo, metricas, alertas_modelo = entrenar()

display(metricas.round(3))
print("Alertas:", alertas_modelo)

,modelo,MAE,RMSE,R2
0,Mediana prueba,49.522,65.006,-0.000
1,XGBoost prueba,48.480,63.220,0.054
2,XGBoost entrenamiento,48.791,63.886,0.111


Alertas: []


## Simular escenarios financieros con Monte Carlo

Se generan múltiples trayectorias posibles de ingresos y gastos.

Para cada mes se calcula:

saldo = saldo anterior + ingresos - gastos - pago de deuda

El ingreso mensual puede proporcionarse directamente. Si se omite,
se utiliza el ingreso anual dividido entre doce como supuesto.

El gasto mensual, saldo inicial y pago mensual de deuda se proporcionan
explícitamente: no están disponibles en el resumen como esos conceptos.

Se utilizan distribuciones lognormales, con independencia entre meses
y entre ingresos y gastos. Las variaciones son supuestos configurables,
no parámetros aprendidos del historial.

Se muestran los percentiles 10, 50 y 90 y la proporción de escenarios
con saldo negativo al cierre de cada mes.

El umbral de alertamiento es ilustrativo y configurable.

In [11]:
# Simular escenarios y registrar resultados

def simular_cliente(
    cliente_id,
    gasto_mensual,
    saldo_inicial,
    pago_deuda_mensual,
    ingreso_mensual=None,
    meses=3,
    escenarios=10000,
    variacion_ingreso=0.10,
    variacion_gasto=0.20,
    umbral_alerta=0.20,
    semilla=42
):
    inicio = perf_counter()
    cliente_id = int(cliente_id)

    try:
        cliente = datos.loc[datos["id"] == cliente_id]

        if len(cliente) != 1:
            raise ValueError("Cliente inexistente.")

        # Preparar ingreso base

        usa_ingreso_anual = ingreso_mensual is None

        ingreso = (
            float(cliente.iloc[0]["yearly_income"]) / 12
            if usa_ingreso_anual
            else float(ingreso_mensual)
        )

        # Validar parámetros

        valores = [
            ingreso,
            gasto_mensual,
            pago_deuda_mensual,
            variacion_ingreso,
            variacion_gasto
        ]

        if not np.isfinite(
            valores + [saldo_inicial, umbral_alerta]
        ).all():
            raise ValueError("Los parámetros deben ser finitos.")

        if min(valores) < 0 or not 0 <= umbral_alerta <= 1:
            raise ValueError(
                "Revisa importes, variaciones y umbral."
            )

        if (
            not isinstance(meses, int)
            or not isinstance(escenarios, int)
            or not 1 <= meses <= 120
            or not 100 <= escenarios <= 100000
        ):
            raise ValueError(
                "Meses: 1–120; escenarios: 100–100000."
            )

        # Generar valores con la media y variación indicadas

        rng = np.random.default_rng(semilla)

        def generar(media, coeficiente_variacion):
            sigma = np.sqrt(
                np.log1p(coeficiente_variacion ** 2)
            )

            return media * rng.lognormal(
                mean=-0.5 * sigma ** 2,
                sigma=sigma,
                size=(escenarios, meses)
            )

        ingresos = generar(ingreso, variacion_ingreso)
        gastos = generar(gasto_mensual, variacion_gasto)

        # Calcular saldo acumulado

        saldos = saldo_inicial + np.cumsum(
            ingresos - gastos - pago_deuda_mensual,
            axis=1
        )

        # Resumir escenarios

        tabla = pd.DataFrame({
            "mes": np.arange(1, meses + 1),
            "ingreso_p10": np.quantile(ingresos, 0.10, axis=0),
            "ingreso_p50": np.quantile(ingresos, 0.50, axis=0),
            "ingreso_p90": np.quantile(ingresos, 0.90, axis=0),
            "saldo_p10": np.quantile(saldos, 0.10, axis=0),
            "saldo_p50": np.quantile(saldos, 0.50, axis=0),
            "saldo_p90": np.quantile(saldos, 0.90, axis=0),
            "prob_saldo_negativo": (saldos < 0).mean(axis=0)
        })

        # Generar alertas

        alertas = [
            "ESCENARIO_CON_SUPUESTOS_NO_CALIBRADOS"
        ]

        if usa_ingreso_anual:
            alertas.append(
                "INGRESO_SUPUESTO_ANUAL_DIVIDIDO_ENTRE_12"
            )

        if (
            tabla["prob_saldo_negativo"].max()
            >= umbral_alerta
        ):
            alertas.append(
                "SALDO_NEGATIVO_SUPERA_UMBRAL_DEL_ESCENARIO"
            )

        # Registrar supuestos y resultados

        detalle = {
            "naturaleza": (
                "Simulación ilustrativa; no pronóstico validado"
            ),
            "supuestos": {
                "ingreso_mensual": ingreso,
                "gasto_mensual": gasto_mensual,
                "saldo_inicial": saldo_inicial,
                "pago_deuda_mensual": pago_deuda_mensual,
                "variacion_ingreso": variacion_ingreso,
                "variacion_gasto": variacion_gasto,
                "meses": meses,
                "escenarios": escenarios,
                "semilla": semilla,
                "umbral_alerta": umbral_alerta,
                "distribucion": (
                    "lognormal; independencia entre meses "
                    "e ingresos/gastos"
                )
            },
            "resultados": tabla.to_dict(orient="records"),
            "alertas": alertas,
            "duracion_segundos": perf_counter() - inicio
        }

        registrar(
            "simulacion",
            "ALERTA",
            detalle,
            cliente_id
        )

        return tabla, alertas

    except Exception as error:
        registrar(
            "simulacion",
            "ERROR",
            {
                "tipo_error": type(error).__name__,
                "mensaje": str(error)
            },
            cliente_id
        )
        raise


# Ejemplo didáctico:
# Estos tres importes NO se obtuvieron del cliente.
# Reemplazarlos por valores proporcionados o supuestos acordados.

simulacion, alertas_simulacion = simular_cliente(
    cliente_id=0,
    gasto_mensual=3000.0,
    saldo_inicial=1000.0,
    pago_deuda_mensual=500.0
)

display(simulacion.round(3))
print("Alertas:", alertas_simulacion)

,mes,ingreso_p10,ingreso_p50,ingreso_p90,saldo_p10,saldo_p50,saldo_p90,prob_saldo_negativo
0,1,4346.866,4947.122,5630.033,1466.191,2506.599,3464.552,0.003
1,2,4363.067,4946.363,5630.150,2541.034,3987.509,5361.511,0.001
2,3,4340.894,4941.933,5624.160,3672.210,5451.617,7157.675,0.000


Alertas: ['ESCENARIO_CON_SUPUESTOS_NO_CALIBRADOS', 'INGRESO_SUPUESTO_ANUAL_DIVIDIDO_ENTRE_12']


## Consultar observabilidad y resultados por cliente

La tabla eventos_modelaje conserva el historial de ejecuciones.

Cada evento incluye fecha, estado, versión de los datos y detalles
de las métricas o simulaciones.

La consulta por cliente recupera su última simulación completada,
incluyendo los supuestos y las alertas necesarios para interpretarla.

Para utilizarla desde el asistente, el identificador debe provenir
de la sesión autenticada.

In [12]:
# Consultar el registro de ejecuciones

with sqlite3.connect(BD) as conexion:
    historial = pd.read_sql_query(
        """
        SELECT
            fecha_utc,
            evento,
            estado,
            cliente_id,
            detalle_json
        FROM eventos_modelaje
        ORDER BY fecha_utc DESC
        LIMIT 10
        """,
        conexion
    )

display(historial)


# La aplicación debe pasar el ID de la sesión autenticada

def consultar_ultima_simulacion(cliente_id_sesion):
    with sqlite3.connect(BD) as conexion:
        fila = conexion.execute(
            """
            SELECT fecha_utc, detalle_json
            FROM eventos_modelaje
            WHERE evento = 'simulacion'
              AND estado != 'ERROR'
              AND cliente_id = ?
            ORDER BY fecha_utc DESC
            LIMIT 1
            """,
            (int(cliente_id_sesion),)
        ).fetchone()

    if fila is None:
        return None

    return {
        "fecha_utc": fila[0],
        **json.loads(fila[1])
    }

,fecha_utc,evento,estado,cliente_id,detalle_json
0,2026-10-22T12:26:53.819205+00:00,simulacion,ALERTA,0,"{""naturaleza"": ""Simulación ilustrativa; no pro..."
1,2026-10-22T12:26:53.651127+00:00,entrenamiento,OK,None,"{""objetivo"": ""credit_score registrado; no prob..."
2,2026-10-22T12:22:28.662716+00:00,simulacion,ALERTA,0,"{""naturaleza"": ""Simulación ilustrativa; no pro..."
3,2026-10-22T12:22:28.453266+00:00,entrenamiento,OK,None,"{""objetivo"": ""credit_score registrado; no prob..."
4,2026-10-22T12:19:15.759200+00:00,simulacion,ALERTA,0,"{""naturaleza"": ""Simulación ilustrativa; no pro..."
5,2026-10-22T12:19:15.571246+00:00,entrenamiento,OK,None,"{""objetivo"": ""credit_score registrado; no prob..."
